In [17]:
#pip install openpyxl

In [1]:
import networkx as nx
import numpy as np
from collections import Counter
from gwgraphs import *
import pandas as pd

In [2]:
def read_graph_data():
    # Read the adjacency matrix (edges)
    with open("COX2_A.txt", "r") as f:
        edges = [tuple(map(int, line.split(','))) for line in f]

    # Read graph indicators
    with open("COX2_graph_indicator.txt", "r") as f:
        graph_indicators = [int(line.strip()) for line in f if line.strip()]

    # Read graph labels
    with open("COX2_graph_labels.txt", "r") as f:
        graph_labels = [int(line.strip()) for line in f]

    # Read node labels
    with open("COX2_node_labels.txt", "r") as f:
        node_labels = [int(line.strip()) for line in f]

    # Read node attributes
    with open("COX2_node_attributes.txt", "r") as f:
        node_attributes = [
            list(map(float,
                     line.strip().split(','))) for line in f if line.strip()
        ]

    return edges, graph_indicators, graph_labels, node_labels, node_attributes


def normalize_features(features):
    features = np.array(features)
    # Normalize each feature dimension independently
    mean = np.mean(features, axis=0)
    std = np.std(features, axis=0)
    # Avoid division by zero
    std[std == 0] = 1
    normalized_features = (features - mean) / std
    return normalized_features.tolist()

def construct_networkx_graph(edges, graph_indicators, graph_labels,
                             node_labels, node_attributes):
    graphs = {}
    graphs_normalized = {}
    
    # Normalize all node attributes
    normalized_attributes = normalize_features(node_attributes)
    
    for node_id, graph_id in enumerate(graph_indicators, start=1):
        if graph_id not in graphs:
            graphs[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])
            graphs_normalized[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])

    for edge in edges:
        node1, node2 = edge
        graph_id = graph_indicators[
            node1 - 1]  # Determine which graph the edge belongs to
        graphs[graph_id].add_edge(node1, node2)
        graphs_normalized[graph_id].add_edge(node1, node2)

    for node_id, (graph_id, node_label,
                  node_attr, norm_attr) in enumerate(zip(graph_indicators, node_labels,
                                              node_attributes, normalized_attributes),
                                          start=1):
        # Original graphs with original features
        graphs[graph_id].nodes[node_id]['label'] = node_label
        graphs[graph_id].nodes[node_id]['attributes'] = node_attr
        
        # Normalized graphs with normalized features
        graphs_normalized[graph_id].nodes[node_id]['label'] = node_label
        graphs_normalized[graph_id].nodes[node_id]['attributes'] = norm_attr

    return graphs, graphs_normalized


def compute_bzr_distance_matrix(graph):
    num_nodes = len(graph.nodes)
    distance_matrix = np.full((num_nodes, num_nodes), 10000.0)

    # Get the correct node indices to be used for indexing the matrix
    node_list = list(graph.nodes())
    node_index = {node: i for i, node in enumerate(node_list)}

    for node in graph.nodes:
        lengths = nx.shortest_path_length(graph, source=node, weight="weight")

        for target, length in lengths.items():
            if target in node_index:
                distance_matrix[node_index[node], node_index[target]] = length

    return distance_matrix


def read_family_labels():
  # Read family labels from the Excel file
  df = pd.read_excel("COX2.xlsx")
  # Extract the "Fam." column and convert it to a list
  family_labels = df["Family"].tolist()
  return family_labels

In [3]:
edges, graph_indicators, graph_labels, node_labels, node_attributes = read_graph_data(
)
graphs, graphs_normalized = construct_networkx_graph(edges, graph_indicators, graph_labels,
                                                   node_labels, node_attributes)
for graph_id, graph in graphs.items():
    print(
        f"Graph ID: {graph_id}, number of nodes: {graph.number_of_nodes()}, label: {graph.graph['label']}"
    )

Graph ID: 1, number of nodes: 39, label: -1
Graph ID: 2, number of nodes: 36, label: -1
Graph ID: 3, number of nodes: 36, label: 1
Graph ID: 4, number of nodes: 39, label: -1
Graph ID: 5, number of nodes: 42, label: -1
Graph ID: 6, number of nodes: 42, label: -1
Graph ID: 7, number of nodes: 44, label: -1
Graph ID: 8, number of nodes: 39, label: -1
Graph ID: 9, number of nodes: 42, label: 1
Graph ID: 10, number of nodes: 42, label: 1
Graph ID: 11, number of nodes: 45, label: 1
Graph ID: 12, number of nodes: 49, label: 1
Graph ID: 13, number of nodes: 42, label: 1
Graph ID: 14, number of nodes: 35, label: -1
Graph ID: 15, number of nodes: 44, label: 1
Graph ID: 16, number of nodes: 47, label: 1
Graph ID: 17, number of nodes: 45, label: 1
Graph ID: 18, number of nodes: 39, label: 1
Graph ID: 19, number of nodes: 44, label: -1
Graph ID: 20, number of nodes: 44, label: -1
Graph ID: 21, number of nodes: 51, label: -1
Graph ID: 22, number of nodes: 45, label: -1
Graph ID: 23, number of nodes

In [4]:
graphs_list = []
labels_list = []
for graph_id, graph in graphs.items():
    # Add the graph object to the list
    graphs_list.append(graph)

    # Extract the label and add to the label list
    label = graph.graph['label']
    labels_list.append(label)

# Convert graphs into pairwise distance matrices
distance_matrices = [compute_bzr_distance_matrix(graph) for graph in graphs_list]
# Determine number of elements of each class
print("Elements of Class 1: ", labels_list.count(1))
print("Elements of Class -1: ", labels_list.count(-1))

Elements of Class 1:  102
Elements of Class -1:  365


In [5]:
## COX2 Ligand Family Information ##
family_labels = read_family_labels()
#print("Family Labels:", family_labels)

In [6]:
# Count occurrences of each family label
family_counts = Counter(family_labels)
print("Original family label counts:", dict(family_counts))

# Keep only families with 10 or more elements
valid_families = {family for family, count in family_counts.items() if count >= 10}
print("Families with 10+ elements:", valid_families)

# Create indices of graphs to keep (where family label is in valid_families)
valid_indices = [i for i, family in enumerate(family_labels) if family in valid_families]

# Filter family labels
filtered_family_labels = [family_labels[i] for i in valid_indices]

# Create filtered graph lists
graphs_list_fam = [graphs_list[i] for i in valid_indices]
graphs_list_fam_normalized = [list(graphs_normalized.values())[i] for i in valid_indices]

# Print remaining family labels with counts
remaining_counts = Counter(filtered_family_labels)
print("Remaining family labels with counts:")
for family, count in remaining_counts.items():
    print(f"  Family {family}: {count} elements")

Original family label counts: {'A.1': 34, 'A.2': 1, 'B.1': 93, 'B.2': 21, 'B.3': 8, 'B.4': 17, 'C': 42, 'D.1': 44, 'D.2': 4, 'E.1': 96, 'E.2': 14, 'F': 31, 'G': 2, 'H': 2, 'I': 1, 'J.1': 36, 'J.2': 2, 'K': 10, 'L': 8, 'M': 1}
Families with 10+ elements: {'B.4', 'B.2', 'A.1', 'K', 'C', 'B.1', 'E.1', 'F', 'J.1', 'D.1', 'E.2'}
Remaining family labels with counts:
  Family A.1: 34 elements
  Family B.1: 93 elements
  Family B.2: 21 elements
  Family B.4: 17 elements
  Family C: 42 elements
  Family D.1: 44 elements
  Family E.1: 96 elements
  Family E.2: 14 elements
  Family F: 31 elements
  Family J.1: 36 elements
  Family K: 10 elements


In [7]:
# See if there is a difference of proportion of active ligands across family groups #
from scipy.stats import chi2_contingency
from itertools import combinations

# Get labels for filtered graphs
filtered_labels = [labels_list[i] for i in valid_indices]

# Create a contingency table: rows are families, columns are classes (active=1, inactive=-1)
print("\n=== Chi-Square Test for Active Ligands Across Families ===")

# Get unique families
unique_families = sorted(set(filtered_family_labels))

# Build contingency table
contingency_table = []
family_names = []

for family in unique_families:
    # Get labels for this family
    family_mask = [i for i, f in enumerate(filtered_family_labels) if f == family]
    family_labels_subset = [filtered_labels[i] for i in family_mask]
    
    # Count active (1) and inactive (-1)
    active_count = family_labels_subset.count(1)
    inactive_count = family_labels_subset.count(-1)
    
    contingency_table.append([active_count, inactive_count])
    family_names.append(family)

# Convert to numpy array for chi-square test
contingency_table = np.array(contingency_table)

print("\nContingency Table (rows=families, cols=[active, inactive]):")
print("Family | Active | Inactive | Total | % Active")
print("-" * 50)
for i, family in enumerate(family_names):
    total = contingency_table[i].sum()
    pct_active = (contingency_table[i, 0] / total) * 100 if total > 0 else 0
    print(f"{family:6} | {contingency_table[i, 0]:6} | {contingency_table[i, 1]:8} | {total:5} | {pct_active:6.2f}%")

# Perform chi-square test
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(f"\nChi-Square Test Results:")
print(f"Chi-square statistic: {chi2:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"P-value: {p_value: .5e}")

alpha = 0.05
if p_value < alpha:
    print(f"\nResult: SIGNIFICANT difference (p < {alpha})")
    print("There is a statistically significant difference in the proportion of active ligands across families.")
    
    # Perform post-hoc pairwise chi-square tests
    print("\n=== Post-hoc Pairwise Chi-Square Tests ===")
    print("(with Bonferroni correction)")
    
    # Calculate number of comparisons for Bonferroni correction
    n_comparisons = len(list(combinations(range(len(family_names)), 2)))
    bonferroni_alpha = alpha / n_comparisons
    
    print(f"Number of pairwise comparisons: {n_comparisons}")
    print(f"Bonferroni-corrected alpha: {bonferroni_alpha:.6f}\n")
    
    significant_pairs = []
    
    from scipy.stats import fisher_exact
    
    for i, j in combinations(range(len(family_names)), 2):
        # Create 2x2 contingency table for this pair
        pair_table = np.array([contingency_table[i], contingency_table[j]])
        
        # Calculate proportions
        prop_i = contingency_table[i, 0] / contingency_table[i].sum()
        prop_j = contingency_table[j, 0] / contingency_table[j].sum()
        
        # Check if chi-square test assumptions are met
        # Use Fisher's exact test if expected frequencies are too small or if there are zeros
        try:
            chi2_pair, p_pair, _, expected_freq = chi2_contingency(pair_table)
            # Check if expected frequencies are all >= 5 (common rule of thumb)
            if np.any(expected_freq < 5):
                # Use Fisher's exact test instead
                _, p_pair = fisher_exact(pair_table)
                test_used = "Fisher"
            else:
                test_used = "Chi-sq"
        except ValueError:
            # If chi-square fails (e.g., zero expected frequencies), use Fisher's exact test
            _, p_pair = fisher_exact(pair_table)
            chi2_pair = np.nan
            test_used = "Fisher"
        
        is_significant = p_pair < bonferroni_alpha
        if is_significant:
            significant_pairs.append((family_names[i], family_names[j], p_pair, prop_i, prop_j))
        
        significance_marker = "***" if is_significant else ""
        if test_used == "Chi-sq":
            print(f"Family {family_names[i]} vs {family_names[j]}: "
                  f"chi2={chi2_pair:.4f}, p={p_pair:.5e} {significance_marker}")
        else:
            print(f"Family {family_names[i]} vs {family_names[j]}: "
                  f"Fisher's exact p={p_pair:.5e} {significance_marker}")
        print(f"  Proportion active: {family_names[i]}={prop_i:.3f}, {family_names[j]}={prop_j:.3f}")
    
    if significant_pairs:
        print(f"\n=== Summary: {len(significant_pairs)} Significant Pair(s) Found ===")
        for fam_i, fam_j, p, prop_i, prop_j in significant_pairs:
            print(f"Family {fam_i} ({prop_i:.1%} active) vs Family {fam_j} ({prop_j:.1%} active): p={p:.5e}")
    else:
        print("\nNo significant pairwise differences after Bonferroni correction.")
else:
    print(f"\nResult: NO significant difference (p >= {alpha})")
    print("There is no statistically significant difference in the proportion of active ligands across families.")


=== Chi-Square Test for Active Ligands Across Families ===

Contingency Table (rows=families, cols=[active, inactive]):
Family | Active | Inactive | Total | % Active
--------------------------------------------------
A.1    |     14 |       20 |    34 |  41.18%
B.1    |     18 |       75 |    93 |  19.35%
B.2    |     10 |       11 |    21 |  47.62%
B.4    |      2 |       15 |    17 |  11.76%
C      |      6 |       36 |    42 |  14.29%
D.1    |      5 |       39 |    44 |  11.36%
E.1    |     33 |       63 |    96 |  34.38%
E.2    |      4 |       10 |    14 |  28.57%
F      |      1 |       30 |    31 |   3.23%
J.1    |      3 |       33 |    36 |   8.33%
K      |      3 |        7 |    10 |  30.00%

Chi-Square Test Results:
Chi-square statistic: 39.8039
Degrees of freedom: 10
P-value:  1.83464e-05

Result: SIGNIFICANT difference (p < 0.05)
There is a statistically significant difference in the proportion of active ligands across families.

=== Post-hoc Pairwise Chi-Square Tests ==

In [8]:
distance_matrices_fam = [compute_bzr_distance_matrix(graph) for graph in graphs_list_fam]

In [18]:
def fgw_knn_alpha_multi(distance_matrices,
                        graphs_list,
                        labels,
                        num_folds,
                        k_list,
                        method,
                        alpha,
                        seed=42,
                        log_knn=False):
    """Function to perform k-nearest neighbors classification using Fused Gromov-Wasserstein distances for multi-class problems.
    Parameters:
    - distance_matrices: list of pairwise distance matrices
    - graphs_list: list of graphs
    - labels: list of labels for each graph
    - num_folds: number of folds for cross-validation
    - k_list: list of k values to test
    - method: cross-validation method, either "kfold" (for k-fold
    CV) or "stratk" (for stratified k-fold CV)
    - alpha: alpha parameter for fGW
    - seed: random seed for reproducibility
    - log_knn: boolean flag to print additional information

    Returns:
    - class_acc_dict: dictionary storing accuracies for each k
    - by_class_acc_dict: dictionary storing accuracies by class for each k
    - f1_score_dict: dictionary storing F1 scores for each k
    - best_k_info: dictionary storing information about the best k value
    """
    # Get unique classes from the labels
    unique_classes = sorted(list(set(labels)))
    print(f"Multi-class classification with {len(unique_classes)} classes: {unique_classes}")
    
    class_acc_dict = {
        k: []
        for k in k_list
    }  # dictionary to store accuracies for each k
    
    # Initialize by_class_acc_dict with all unique classes
    by_class_acc_dict = {
        k: {class_label: [] for class_label in unique_classes}
        for k in k_list
    }  # store accuracies by class for each k

    f1_score_dict = {
        k: []
        for k in k_list
    }  # dictionary to store F1 scores for each k

    assert method in ["kfold", "stratk"]
    if method == "kfold":
        k_folds = KFold(n_splits=num_folds, shuffle=True, random_state=seed)
        print("k-fold CV selected")
    else:
        k_folds = StratifiedKFold(n_splits=num_folds,
                                  shuffle=True,
                                  random_state=seed)
        print("Stratified k-fold CV selected")
    start = time.time()

    for i, (train_index,
            test_index) in enumerate(k_folds.split(distance_matrices, labels)):
        print(f"Fold {i}:")
        if log_knn:
            print(f"  Train: index={train_index}")
            print(f"  Test:  index={test_index}")

        X_train = [distance_matrices[idx] for idx in train_index]
        y_train = [labels[idx] for idx in train_index]
        X_test = [distance_matrices[idx] for idx in test_index]
        y_test = [labels[idx] for idx in test_index]

        all_distances = []
        for test_idx, (test_graph,
                       ground_truth_label) in enumerate(zip(X_test, y_test)):
            # Get test graph features
            test_graph_obj = graphs_list[test_index[test_idx]]
            test_features = np.array([
                test_graph_obj.nodes[n]['attributes']
                for n in test_graph_obj.nodes()
            ])

            distances = []
            for train_idx, (train_graph,
                            train_label) in enumerate(zip(X_train, y_train)):
                # Get train graph features
                train_graph_obj = graphs_list[train_index[train_idx]]
                train_features = np.array([
                    train_graph_obj.nodes[n]['attributes']
                    for n in train_graph_obj.nodes()
                ])

                # Compute metric cost matrix between test and train features
                M = ot.dist(test_features, train_features)

                # Compute fGW distance using the M for this pair
                fgw, log = ot.gromov.fused_gromov_wasserstein(
                    M,
                    test_graph,
                    train_graph,
                    loss_fun='square_loss',
                    alpha=alpha,
                    log=True,
                    random_seed=seed)
                distances.append((log['fgw_dist'], train_label))
            distances.sort(key=lambda x: x[0])  # Sort based on distances
            all_distances.append((distances, ground_truth_label))

        # For each k value, perform classification
        for k in k_list:
            correct_count = 0
            class_correct = {class_label: 0 for class_label in unique_classes}
            class_total = {class_label: 0 for class_label in unique_classes}
            y_true = []
            y_pred = []

            for distances, ground_truth_label in all_distances:
                # Take the k nearest neighbors
                k_nearest_neighbors = distances[:k]
                # Determine the most common label
                labels_counts = {}
                for _, label in k_nearest_neighbors:
                    labels_counts[label] = labels_counts.get(label, 0) + 1
                predicted_label = max(labels_counts, key=labels_counts.get)

                y_true.append(ground_truth_label)
                y_pred.append(predicted_label)

                if predicted_label == ground_truth_label:
                    correct_count += 1
                    class_correct[ground_truth_label] += 1
                class_total[ground_truth_label] += 1

            # Calculate overall and by-class accuracies
            classification_accuracy = (correct_count / len(X_test)) * 100
            class_acc_dict[k].append(classification_accuracy)

            # Calculate F1 score with appropriate averaging
            if method == "kfold":
                f1 = f1_score(y_true, y_pred, average='macro') * 100
            else:  # stratk
                f1 = f1_score(y_true, y_pred, average='weighted') * 100
            f1_score_dict[k].append(f1)

            # Calculate and store by-class accuracies for all classes
            for class_label in unique_classes:
                if class_total[class_label] > 0:
                    class_accuracy = (class_correct[class_label] /
                                      class_total[class_label]) * 100
                    by_class_acc_dict[k][class_label].append(class_accuracy)
                    print(
                        f"k={k}, Class {class_label} Accuracy: {class_accuracy}%, Fold {i}"
                    )

            print(
                f"k={k}, Overall Classification Accuracy: {classification_accuracy}%, Fold {i}"
            )
            average_type = "macro" if method == "kfold" else "weighted"
            print(f"k={k}, F1 Score ({average_type} avg): {f1:.2f}%, Fold {i}")

    end = time.time()
    print("Total runtime, seconds:", end - start)

    # Print accuracy stats for each k and print best overall k
    best_k = k_list[0]
    best_acc = 0
    best_f1 = 0
    k_avg_accuracies = {}
    k_avg_f1_scores = {}
    k_std_accuracies = {}
    k_std_f1_scores = {}

    average_type = "macro" if method == "kfold" else "weighted"

    for k in k_list:
        avg_acc = sum(class_acc_dict[k]) / len(class_acc_dict[k])
        avg_f1 = sum(f1_score_dict[k]) / len(f1_score_dict[k])

        # Calculate standard deviations
        std_acc = np.std(class_acc_dict[k])
        std_f1 = np.std(f1_score_dict[k])
        
        # Build accuracy dictionary for this k
        k_avg_accuracies[k] = {'overall': avg_acc}
        k_std_accuracies[k] = {'overall': std_acc}
        
        # Calculate average and std for each class
        for class_label in unique_classes:
            if by_class_acc_dict[k][class_label]:  # Check if list is not empty
                class_avg = sum(by_class_acc_dict[k][class_label]) / len(by_class_acc_dict[k][class_label])
                class_std = np.std(by_class_acc_dict[k][class_label])
                k_avg_accuracies[k][f'class_{class_label}'] = class_avg
                k_std_accuracies[k][f'class_{class_label}'] = class_std
        
        k_avg_f1_scores[k] = avg_f1
        k_std_f1_scores[k] = std_f1

        print(f"\nk={k} Average Overall Accuracy: {avg_acc:.2f}%")
        print(f"k={k} Average F1 Score ({average_type} avg): {avg_f1:.2f}%")
        for class_label in unique_classes:
            if f'class_{class_label}' in k_avg_accuracies[k]:
                print(
                    f"k={k} Average Class {class_label} Accuracy: {k_avg_accuracies[k][f'class_{class_label}']:.2f}%"
                )

        if avg_acc > best_acc:
            best_acc = avg_acc
            best_k = k

        if avg_f1 > best_f1:
            best_f1 = avg_f1

    print(f"\nBest performance achieved with k={best_k}:")
    print(
        f"Overall Accuracy: {k_avg_accuracies[best_k]['overall']:.2f}% ± {k_std_accuracies[best_k]['overall']:.2f}%"
    )
    print(
        f"F1 Score ({average_type} avg): {k_avg_f1_scores[best_k]:.2f}% ± {k_std_f1_scores[best_k]:.2f}%"
    )
    
    # Print best accuracies for each class
    for class_label in unique_classes:
        if f'class_{class_label}' in k_avg_accuracies[best_k]:
            print(
                f"Class {class_label} Accuracy: {k_avg_accuracies[best_k][f'class_{class_label}']:.2f}% ± {k_std_accuracies[best_k][f'class_{class_label}']:.2f}%"
            )

    # Prepare best k information to return - generalize for any number of classes
    best_k_info = {
        'best_k': best_k,
        'best_overall_accuracy': k_avg_accuracies[best_k]['overall'],
        'best_f1_score': k_avg_f1_scores[best_k]
    }
    
    # Add class-specific accuracies to best_k_info
    for class_label in unique_classes:
        if f'class_{class_label}' in k_avg_accuracies[best_k]:
            best_k_info[f'best_class_{class_label}_accuracy'] = k_avg_accuracies[best_k][f'class_{class_label}']

    return class_acc_dict, by_class_acc_dict, f1_score_dict, best_k_info

In [11]:
k_list = [1,2,3,4,5,6,7,8,9,10]
num_folds = 5

In [20]:
alpha_list = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
alpha_tests_k = {
        alph: {}
        for alph in alpha_list
    }  # dictionary to store best k information for each alpha

for alpha in alpha_list:
    print()
    print("Currently testing alpha =", alpha)
    print()
    class_acc1, by_class_acc1, f1_score1, best_k_info1 = fgw_knn_alpha_multi(distance_matrices_fam,
                        graphs_list_fam,
                        filtered_family_labels,
                        num_folds=num_folds,
                        k_list=k_list,
                        method="stratk",
                        alpha=alpha)
    # Store the best k information for this alpha
    alpha_tests_k[alpha] = best_k_info1


Currently testing alpha = 0.0

Multi-class classification with 11 classes: ['A.1', 'B.1', 'B.2', 'B.4', 'C', 'D.1', 'E.1', 'E.2', 'F', 'J.1', 'K']
Stratified k-fold CV selected
Fold 0:
k=1, Class A.1 Accuracy: 71.42857142857143%, Fold 0
k=1, Class B.1 Accuracy: 94.73684210526315%, Fold 0
k=1, Class B.2 Accuracy: 50.0%, Fold 0
k=1, Class B.4 Accuracy: 33.33333333333333%, Fold 0
k=1, Class C Accuracy: 100.0%, Fold 0
k=1, Class D.1 Accuracy: 88.88888888888889%, Fold 0
k=1, Class E.1 Accuracy: 84.21052631578947%, Fold 0
k=1, Class E.2 Accuracy: 33.33333333333333%, Fold 0
k=1, Class F Accuracy: 83.33333333333334%, Fold 0
k=1, Class J.1 Accuracy: 57.14285714285714%, Fold 0
k=1, Class K Accuracy: 0.0%, Fold 0
k=1, Overall Classification Accuracy: 78.4090909090909%, Fold 0
k=1, F1 Score (weighted avg): 77.35%, Fold 0
k=2, Class A.1 Accuracy: 71.42857142857143%, Fold 0
k=2, Class B.1 Accuracy: 94.73684210526315%, Fold 0
k=2, Class B.2 Accuracy: 50.0%, Fold 0
k=2, Class B.4 Accuracy: 33.3333333

In [21]:
best_alpha = alpha_list[0]
best_alpha_acc = 0
best_alpha_f1 = 0

unique_classes = sorted(list(set(filtered_family_labels)))

for alph in alpha_list:
    best_k_info = alpha_tests_k[alph]
    
    # Extract overall metrics
    overall_acc = best_k_info['best_overall_accuracy']
    f1_score = best_k_info['best_f1_score']
    best_k = best_k_info['best_k']
    
    print(f"\nAlpha = {alph}")
    print(f"  Best k: {best_k}")
    print(f"  Overall Accuracy: {overall_acc:.2f}%")
    print(f"  F1 Score (weighted avg): {f1_score:.2f}%")
    
    # Print class-specific accuracies
    for class_label in unique_classes:
        class_key = f'best_class_{class_label}_accuracy'
        if class_key in best_k_info:
            print(f"  Class {class_label} Accuracy: {best_k_info[class_key]:.2f}%")
    
    # Track best overall performance
    if overall_acc > best_alpha_acc:
        best_alpha_acc = overall_acc
        best_alpha = alph
    
    if f1_score > best_alpha_f1:
        best_alpha_f1 = f1_score

print("\n" + "="*60)
print("BEST ALPHA PERFORMANCE")
print("="*60)
best_k_info = alpha_tests_k[best_alpha]
print(f"Best Alpha: {best_alpha}")
print(f"Best k: {best_k_info['best_k']}")
print(f"Overall Accuracy: {best_k_info['best_overall_accuracy']:.2f}%")
print(f"F1 Score (weighted avg): {best_k_info['best_f1_score']:.2f}%")

# Print best class-specific accuracies
for class_label in unique_classes:
    class_key = f'best_class_{class_label}_accuracy'
    if class_key in best_k_info:
        print(f"Class {class_label} Accuracy: {best_k_info[class_key]:.2f}%")


Alpha = 0.0
  Best k: 1
  Overall Accuracy: 80.36%
  F1 Score (weighted avg): 79.46%
  Class A.1 Accuracy: 82.38%
  Class B.1 Accuracy: 83.80%
  Class B.2 Accuracy: 42.00%
  Class B.4 Accuracy: 58.33%
  Class C Accuracy: 97.78%
  Class D.1 Accuracy: 95.56%
  Class E.1 Accuracy: 86.47%
  Class E.2 Accuracy: 43.33%
  Class F Accuracy: 93.33%
  Class J.1 Accuracy: 66.07%
  Class K Accuracy: 20.00%

Alpha = 0.1
  Best k: 1
  Overall Accuracy: 80.13%
  F1 Score (weighted avg): 79.28%
  Class A.1 Accuracy: 82.38%
  Class B.1 Accuracy: 83.80%
  Class B.2 Accuracy: 42.00%
  Class B.4 Accuracy: 58.33%
  Class C Accuracy: 97.78%
  Class D.1 Accuracy: 93.33%
  Class E.1 Accuracy: 86.47%
  Class E.2 Accuracy: 43.33%
  Class F Accuracy: 93.33%
  Class J.1 Accuracy: 66.07%
  Class K Accuracy: 20.00%

Alpha = 0.2
  Best k: 1
  Overall Accuracy: 80.13%
  F1 Score (weighted avg): 79.36%
  Class A.1 Accuracy: 82.38%
  Class B.1 Accuracy: 83.80%
  Class B.2 Accuracy: 42.00%
  Class B.4 Accuracy: 58.33%


In [12]:
def gw_knn_multi(distance_matrices,
                  labels,
                  num_folds,
                  k_list,
                  method,
                  seed=42,
                  log_knn=False):
    """Function to perform k-nearest neighbors classification using Gromov-Wasserstein distances for multi-class problems.
    Parameters:
    - distance_matrices: List of distance matrices.
    - labels: List of labels corresponding to the distance matrices.
    - num_folds: Number of folds for cross-validation.
    - k_list: List of k values to test.
    - method: Cross-validation method, either 'kfold' or 'stratk'.
    - seed: Random seed for reproducibility. Defaults to 42.
    - log_knn: Boolean flag to log k-NN details. Defaults to False.

    Returns:
    - class_acc_dict: Dictionary storing accuracies for each k.
    - by_class_acc_dict: Dictionary storing accuracies by class for each k.
    - f1_score_dict: Dictionary storing F1 scores for each k.
    - best_k_info: Dictionary storing information about the best k value.
    """
    # Get unique classes from the labels
    unique_classes = sorted(list(set(labels)))
    print(f"Multi-class classification with {len(unique_classes)} classes: {unique_classes}")
    
    class_acc_dict = {
        k: []
        for k in k_list
    }  # dictionary to store accuracies for each k
    
    # Initialize by_class_acc_dict with all unique classes
    by_class_acc_dict = {
        k: {class_label: [] for class_label in unique_classes}
        for k in k_list
    }  # store accuracies by class for each k

    f1_score_dict = {
        k: []
        for k in k_list
    }  # dictionary to store F1 scores for each k
    assert method in ["kfold", "stratk"]
    if method == "kfold":
        skf = KFold(n_splits=num_folds, shuffle=True, random_state=seed)
        print("k-fold CV selected")
    else:
        skf = StratifiedKFold(n_splits=num_folds,
                              shuffle=True,
                              random_state=seed)
        print("Stratified k-fold CV selected")
    start = time.time()

    for i, (train_index,
            test_index) in enumerate(skf.split(distance_matrices, labels)):
        print(f"Fold {i}:")
        if log_knn:
            print(f"  Train: index={train_index}")
            print(f"  Test:  index={test_index}")

        X_train = [distance_matrices[idx] for idx in train_index]
        y_train = [labels[idx] for idx in train_index]
        X_test = [distance_matrices[idx] for idx in test_index]
        y_test = [labels[idx] for idx in test_index]

        # Compute all pairwise distances once per fold
        all_distances = []
        for test_graph, ground_truth_label in zip(X_test, y_test):
            distances = []
            for train_graph, train_label in zip(X_train, y_train):
                gw, log = ot.gromov.gromov_wasserstein(test_graph,
                                                       train_graph,
                                                       loss_fun='square_loss',
                                                       log=True,
                                                       random_seed=seed)
                distances.append((log['gw_dist'], train_label))
            distances.sort(key=lambda x: x[0])  # Sort based on distances
            all_distances.append((distances, ground_truth_label))

        # For each k value, perform classification
        for k in k_list:
            correct_count = 0
            class_correct = {class_label: 0 for class_label in unique_classes}
            class_total = {class_label: 0 for class_label in unique_classes}
            y_true = []
            y_pred = []

            for distances, ground_truth_label in all_distances:
                # Take the k nearest neighbors
                k_nearest_neighbors = distances[:k]
                # Determine the most common label
                labels_counts = {}
                for _, label in k_nearest_neighbors:
                    labels_counts[label] = labels_counts.get(label, 0) + 1
                predicted_label = max(labels_counts, key=labels_counts.get)

                y_true.append(ground_truth_label)
                y_pred.append(predicted_label)

                if predicted_label == ground_truth_label:
                    correct_count += 1
                    class_correct[ground_truth_label] += 1
                class_total[ground_truth_label] += 1

            # Calculate overall and by-class accuracies
            classification_accuracy = (correct_count / len(X_test)) * 100
            class_acc_dict[k].append(classification_accuracy)

            # Calculate F1 score with appropriate averaging
            if method == "kfold":
                f1 = f1_score(y_true, y_pred, average='macro') * 100
            else:  # stratk
                f1 = f1_score(y_true, y_pred, average='weighted') * 100
            f1_score_dict[k].append(f1)

            # Calculate and store by-class accuracies for all classes
            for class_label in unique_classes:
                if class_total[class_label] > 0:
                    class_accuracy = (class_correct[class_label] /
                                      class_total[class_label]) * 100
                    by_class_acc_dict[k][class_label].append(class_accuracy)
                    print(
                        f"k={k}, Class {class_label} Accuracy: {class_accuracy}%, Fold {i}"
                    )

            print(
                f"k={k}, Overall Classification Accuracy: {classification_accuracy}%, Fold {i}"
            )
            average_type = "macro" if method == "kfold" else "weighted"
            print(f"k={k}, F1 Score ({average_type} avg): {f1:.2f}%, Fold {i}")

    end = time.time()
    print("Total runtime, seconds:", end - start)

    # Print accuracy stats for each k and print best overall k
    best_k = k_list[0]
    best_acc = 0
    best_f1 = 0
    k_avg_accuracies = {}
    k_avg_f1_scores = {}
    k_std_accuracies = {}
    k_std_f1_scores = {}

    average_type = "macro" if method == "kfold" else "weighted"

    for k in k_list:
        avg_acc = sum(class_acc_dict[k]) / len(class_acc_dict[k])
        avg_f1 = sum(f1_score_dict[k]) / len(f1_score_dict[k])

        # Calculate standard deviations
        std_acc = np.std(class_acc_dict[k])
        std_f1 = np.std(f1_score_dict[k])
        
        # Build accuracy dictionary for this k
        k_avg_accuracies[k] = {'overall': avg_acc}
        k_std_accuracies[k] = {'overall': std_acc}
        
        # Calculate average and std for each class
        for class_label in unique_classes:
            if by_class_acc_dict[k][class_label]:  # Check if list is not empty
                class_avg = sum(by_class_acc_dict[k][class_label]) / len(by_class_acc_dict[k][class_label])
                class_std = np.std(by_class_acc_dict[k][class_label])
                k_avg_accuracies[k][f'class_{class_label}'] = class_avg
                k_std_accuracies[k][f'class_{class_label}'] = class_std
        
        k_avg_f1_scores[k] = avg_f1
        k_std_f1_scores[k] = std_f1

        print(f"\nk={k} Average Overall Accuracy: {avg_acc:.2f}%")
        print(f"k={k} Average F1 Score ({average_type} avg): {avg_f1:.2f}%")
        for class_label in unique_classes:
            if f'class_{class_label}' in k_avg_accuracies[k]:
                print(
                    f"k={k} Average Class {class_label} Accuracy: {k_avg_accuracies[k][f'class_{class_label}']:.2f}%"
                )

        if avg_acc > best_acc:
            best_acc = avg_acc
            best_k = k

        if avg_f1 > best_f1:
            best_f1 = avg_f1

    print(f"\nBest performance achieved with k={best_k}:")
    print(
        f"Overall Accuracy: {k_avg_accuracies[best_k]['overall']:.2f}% ± {k_std_accuracies[best_k]['overall']:.2f}%"
    )
    print(
        f"F1 Score ({average_type} avg): {k_avg_f1_scores[best_k]:.2f}% ± {k_std_f1_scores[best_k]:.2f}%"
    )
    
    # Print best accuracies for each class
    for class_label in unique_classes:
        if f'class_{class_label}' in k_avg_accuracies[best_k]:
            print(
                f"Class {class_label} Accuracy: {k_avg_accuracies[best_k][f'class_{class_label}']:.2f}% ± {k_std_accuracies[best_k][f'class_{class_label}']:.2f}%"
            )

    # Prepare best k information to return - generalize for any number of classes
    best_k_info = {
        'best_k': best_k,
        'best_overall_accuracy': k_avg_accuracies[best_k]['overall'],
        'best_f1_score': k_avg_f1_scores[best_k]
    }
    
    # Add class-specific accuracies to best_k_info
    for class_label in unique_classes:
        if f'class_{class_label}' in k_avg_accuracies[best_k]:
            best_k_info[f'best_class_{class_label}_accuracy'] = k_avg_accuracies[best_k][f'class_{class_label}']

    return class_acc_dict, by_class_acc_dict, f1_score_dict, best_k_info

In [13]:
class_acc2, by_class_acc2, f1_score2, best_k2 = gw_knn_multi(distance_matrices_fam,
                        filtered_family_labels,
                        num_folds=num_folds,
                        k_list=k_list,
                        method="stratk")

Multi-class classification with 11 classes: ['A.1', 'B.1', 'B.2', 'B.4', 'C', 'D.1', 'E.1', 'E.2', 'F', 'J.1', 'K']
Stratified k-fold CV selected
Fold 0:
k=1, Class A.1 Accuracy: 100.0%, Fold 0
k=1, Class B.1 Accuracy: 89.47368421052632%, Fold 0
k=1, Class B.2 Accuracy: 100.0%, Fold 0
k=1, Class B.4 Accuracy: 100.0%, Fold 0
k=1, Class C Accuracy: 100.0%, Fold 0
k=1, Class D.1 Accuracy: 77.77777777777779%, Fold 0
k=1, Class E.1 Accuracy: 57.89473684210527%, Fold 0
k=1, Class E.2 Accuracy: 33.33333333333333%, Fold 0
k=1, Class F Accuracy: 83.33333333333334%, Fold 0
k=1, Class J.1 Accuracy: 85.71428571428571%, Fold 0
k=1, Class K Accuracy: 0.0%, Fold 0
k=1, Overall Classification Accuracy: 79.54545454545455%, Fold 0
k=1, F1 Score (weighted avg): 79.13%, Fold 0
k=2, Class A.1 Accuracy: 100.0%, Fold 0
k=2, Class B.1 Accuracy: 89.47368421052632%, Fold 0
k=2, Class B.2 Accuracy: 100.0%, Fold 0
k=2, Class B.4 Accuracy: 100.0%, Fold 0
k=2, Class C Accuracy: 100.0%, Fold 0
k=2, Class D.1 Accurac